# Wan2GP on Google Colab

Sets up [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) in a fresh GPU-backed Colab session.

Run the cells in order to prepare the runtime, install dependencies, and launch the Gradio interface. Click on the link in the output from the last cell to launch the app in your browser.

> **Colab VRAM note:** the free tier usually assigns a 15 GB T4 GPU. Most Wan2GP models exceed that budget; the Wan 2.2 TextImage2Video FastWan model works, producing roughly a 5 second 480p clip in about 8 minutes.

> **Tip:** lower the resolution in the Wan2GP interface before your first generation. The models default to 1280x720, which is more than a free Colab GPU handles comfortably.


## 1. Confirm the accelerator

Choose `Runtime → Change runtime type` and select **GPU** before running anything else.

If this cell raises an error, go back to `Runtime → Change runtime type`, pick **GPU** and save.


In [1]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime → Change runtime type, select GPU (or TPU if GPUs are unavailable), save, then rerun this cell.'
    ) from exc


## 2. Configure the workspace path and optional persistent data storage


Set `USE_GOOGLE_DRIVE_DATA = True` if you want Google Drive to keep checkpoints, LoRAs, outputs and model caches across Colab restarts.


In [2]:
from pathlib import Path


# CHANGE THIS TO TRUE IF YOU WANT TO USE GOOGLE DRIVE FOR DATA STORAGE (PERSISTENT ACROSS SESSIONS)
USE_GOOGLE_DRIVE_DATA = False




DRIVE_MOUNT_POINT = Path('/content/drive')
WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
EPHEMERAL_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
PERSISTENT_DATA_ROOT = (DRIVE_MOUNT_POINT / 'MyDrive' / 'Wan2GP-data').resolve()

if USE_GOOGLE_DRIVE_DATA:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    WAN_DATA_ROOT = PERSISTENT_DATA_ROOT
    data_mode = 'Google Drive (persistent data)'
else:
    WAN_DATA_ROOT = EPHEMERAL_DATA_ROOT
    data_mode = 'Colab runtime disk (ephemeral data)'

WAN_CKPTS_DIR = (WAN_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_DATA_ROOT / 'loras').resolve()
WAN_OUTPUTS_DIR = (WAN_DATA_ROOT / 'outputs').resolve()
WAN_CACHE_DIR = (WAN_DATA_ROOT / 'cache').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

WAN2GP_ROOT.parent.mkdir(parents=True, exist_ok=True)
WAN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
WAN_CKPTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_22B_LORAS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Wan2GP repository path: {WAN2GP_ROOT}')
print(f'Data storage mode: {data_mode}')
print(f'Data root: {WAN_DATA_ROOT}')
print(f'Checkpoints: {WAN_CKPTS_DIR}')
print(f'LoRAs: {WAN_LORAS_DIR}')
print(f'LTX-2 LoRAs: {WAN_LTX2_LORAS_DIR}')
print(f'LTX-2 22B LoRAs: {WAN_LTX2_22B_LORAS_DIR}')
print(f'Outputs: {WAN_OUTPUTS_DIR}')
print(f'Cache: {WAN_CACHE_DIR}')


Wan2GP repository path: /content/Wan2GP
Data storage mode: Colab runtime disk (ephemeral data)
Data root: /content/Wan2GP-data
Checkpoints: /content/Wan2GP-data/ckpts
LoRAs: /content/Wan2GP-data/loras
LTX-2 LoRAs: /content/Wan2GP-data/loras/ltx2
LTX-2 22B LoRAs: /content/Wan2GP-data/loras/ltx2_22B
Outputs: /content/Wan2GP-data/outputs
Cache: /content/Wan2GP-data/cache


## 3. Download or update Wan2GP

Clone the repository if it is not present yet; otherwise pull the latest changes.


In [3]:
import shutil, subprocess
from pathlib import Path

def merge_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in list(source_dir.iterdir()):
        destination = destination_dir / child.name
        if destination.exists():
            if child.is_dir() and destination.is_dir():
                merge_directory_contents(child, destination)
                child.rmdir()
                continue
            raise RuntimeError(f'Cannot move {child} into {destination_dir}: {destination} already exists.')
        shutil.move(str(child), str(destination))

def attach_data_directory(repo_path: Path, data_path: Path) -> None:
    data_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        if repo_path.resolve() != data_path.resolve():
            raise RuntimeError(f'{repo_path} already points to {repo_path.resolve()}, expected {data_path}.')
        print(f'Using existing link: {repo_path} -> {data_path}')
        return

    if repo_path.exists():
        if not repo_path.is_dir():
            raise RuntimeError(f'Expected a directory at {repo_path}.')
        merge_directory_contents(repo_path, data_path)
        repo_path.rmdir()
    else:
        repo_path.parent.mkdir(parents=True, exist_ok=True)

    repo_path.symlink_to(data_path, target_is_directory=True)
    print(f'Linked {repo_path.name} -> {data_path}')

MANAGED_PATHS = {'ckpts', 'loras', 'outputs', 'ffmpeg_bins'}

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    untracked = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'ls-files', '--others', '--exclude-standard'],
        check=True,
        capture_output=True,
        text=True,
    )
    user_files = [
        line for line in untracked.stdout.splitlines()
        if line.split('/', 1)[0] not in MANAGED_PATHS
    ]
    if user_files:
        preview = ', '.join(user_files[:5]) + (' …' if len(user_files) > 5 else '')
        print(f'Repository has your own files in it ({preview}). Skipping the update to keep them.')
    else:
        print('Repository already exists. Updating to the latest version...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'checkout', '--', '.'], check=True)
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

attach_data_directory(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
attach_data_directory(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
attach_data_directory(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)


Linked ckpts -> /content/Wan2GP-data/ckpts
Linked loras -> /content/Wan2GP-data/loras
Linked outputs -> /content/Wan2GP-data/outputs


## 4. Install system dependencies

Install shared libraries needed for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.

In [4]:
import os, shutil, subprocess

APT_PACKAGES = ['ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2']

def dpkg_installed(package: str) -> bool:
    return subprocess.run(['dpkg', '-s', package], capture_output=True).returncode == 0

missing = [package for package in APT_PACKAGES if not dpkg_installed(package)]
if 'ffmpeg' in missing and shutil.which('ffmpeg'):
    missing.remove('ffmpeg')

if missing:
    env = os.environ.copy()
    env['DEBIAN_FRONTEND'] = 'noninteractive'
    print('Installing:', ', '.join(missing))
    subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
    subprocess.run([
        'sudo', 'apt-get', 'install', '-y', '--no-install-recommends', *missing
    ], check=True, env=env)
else:
    print('System dependencies already installed.')


Installing: libportaudio2


## 5. Install Python dependencies

Install PyTorch and Wan2GP's Python packages. This takes a few minutes.

In [5]:
import json, os, subprocess, sys

PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
REPLACEMENT_TORCH = ('torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0')
UNSUPPORTED_TORCH = ('2.8.0', '2.9.')
ONNXRUNTIME_GPU = 'onnxruntime-gpu==1.22.0'

TORCH_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import importlib.metadata as md, torch\n'
    '    for name in ("torch", "torchvision", "torchaudio"):\n'
    '        try:\n'
    '            info[name] = md.version(name)\n'
    '        except Exception:\n'
    '            pass\n'
    '    info["cuda"] = torch.cuda.is_available()\n'
    '    if info["cuda"]:\n'
    '        info["device"] = torch.cuda.get_device_name(0)\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

ONNX_PROBE = (
    'import json\n'
    'info = {}\n'
    'try:\n'
    '    import onnxruntime\n'
    '    info["version"] = onnxruntime.__version__\n'
    '    info["providers"] = onnxruntime.get_available_providers()\n'
    'except Exception:\n'
    '    pass\n'
    'print(json.dumps(info))\n'
)

env = os.environ.copy()

if USE_GOOGLE_DRIVE_DATA:
    uv_cache_dir = WAN_CACHE_DIR / 'uv'
    uv_cache_dir.mkdir(parents=True, exist_ok=True)
    env['UV_CACHE_DIR'] = str(uv_cache_dir)
    env['UV_LINK_MODE'] = 'copy'

def uv_pip(*args):
    subprocess.run([sys.executable, '-m', 'uv', 'pip', *args], check=True, env=env)

def uv_install(*args):
    uv_pip('install', '--system', *args)

def probe(code):
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, env=env)
    try:
        return json.loads(result.stdout.strip().splitlines()[-1])
    except Exception:
        return {}

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv'], check=True, env=env)

torch_version = probe(TORCH_PROBE).get('torch', '')
try:
    torch_release = tuple(int(part) for part in torch_version.split('.')[:2])
except ValueError:
    torch_release = ()
torch_usable = (
    torch_release >= (2, 7)
    and not torch_version.startswith(UNSUPPORTED_TORCH)
)

if torch_usable:
    print(f'Using the PyTorch already installed in this runtime ({torch_version}).')
else:
    print('Installing PyTorch...')
    uv_install(*REPLACEMENT_TORCH, '--index-url', PYTORCH_INDEX_URL)

print('Installing Wan2GP packages...')
uv_install('-r', str(WAN2GP_ROOT / 'requirements.txt'), '--index-strategy', 'unsafe-best-match')

uv_install(ONNXRUNTIME_GPU)
onnx = probe(ONNX_PROBE)
if not onnx.get('version'):
    uv_pip('uninstall', '--system', 'onnxruntime-gpu')
    uv_install('onnxruntime>=1.22.0')
    onnx = probe(ONNX_PROBE)

report = probe(TORCH_PROBE)
if not report.get('torch'):
    raise RuntimeError(
        'PyTorch is not working after installation. Open Runtime -> Restart session, then run Steps 2 to 5 again.'
    )
if not report.get('cuda'):
    raise RuntimeError(
        'No GPU detected. Open Runtime -> Change runtime type, choose GPU, then run Steps 1 and 5 again.'
    )

print(f"Ready: PyTorch {report['torch']} on {report['device']}.")
if 'CUDAExecutionProvider' not in onnx.get('providers', []):
    print('Note: background removal will run on the CPU.')


Using the PyTorch already installed in this runtime (2.11.0+cu128).
Installing Wan2GP packages...
Ready: PyTorch 2.11.0+cu128 on Tesla T4.


## 5b. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend so Step 6 launches cleanly in Colab.


In [6]:
from pathlib import Path

# Replace the TkAgg backend with the headless Agg backend if present.
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


Replaced TkAgg with Agg in interact_tools.py.


## 6. Launch Wan2GP

Run the Gradio interface. You will find the gradio link in the output. Click on the link to access the UI. Keep the cell running to stay connected; stop it with the square **Stop** button when you are finished.

In [ ]:
import os, subprocess, sys, threading, time

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')
cmd = [
    sys.executable,
    '-u',
    'wgp.py',
    '--listen',
    '--server-port', '7860',
    '--share',
    '--profile', '5',
]

if USE_GOOGLE_DRIVE_DATA:
    print('Using Google Drive for checkpoints, LoRAs, outputs and caches.')
else:
    print('Using Colab runtime storage for checkpoints, LoRAs, outputs and caches.')
print('Launching Wan2GP…')
process = subprocess.Popen(
    cmd,
    cwd=str(WAN2GP_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
stop_event = threading.Event()

def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if stop_event.is_set():
            break
        print('[keepalive] Notebook cell still running…')

keepalive_thread = threading.Thread(target=keepalive, daemon=True)
keepalive_thread.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping Wan2GP…')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    keepalive_thread.join(timeout=1)
    print(f'Wan2GP stopped (return code: {process.returncode}).')


Using Colab runtime storage for checkpoints, LoRAs, outputs and caches.
Launching Wan2GP…
[keepalive] Notebook cell still running…
[GGUF][llama.cpp CUDA] kernels unavailable, using fallback
[keepalive] Notebook cell still running…
Switching to FP16 models when possible as GPU architecture doesn't support optimed BF16 Kernels
Loaded plugin: Motion Designer (from motion_designer)
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://9f028a21cba1de6a2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
[keepalive] Notebook cell still running…
[keepalive] Notebook cell still running…
[keepalive] Notebook cell still running…
[keepalive] Notebook cell still running…
[keepalive] Notebook cell still running…
[keepalive] Notebook cell still running…
[keepalive] Notebook cell still running…
[keepalive] No